# Paper 4, revision runs of the open models

Choose an **L4** runtime (Runtime, Change runtime type), then Runtime, Run all. Before that, upload `revision_upload.zip` to `My Drive/Jev/paper4/colab/` (the folder of the first run). The notebook unpacks it, checks every file against its SHA-256, and runs:

1. each open decision model on the eight permutation conditions `d2_k{5,20,50,150}_p{2,3}` (rep 1);
2. this-that-1.0 with description-only option text (`this-that-1.0-desc`) on `d1_neutral` and `d2_k150`;
3. a rep-2 retest of each open model on `e2_d3_conv_go_awry_kny` and `e2_d3_wiki_corpus_kny` (switch `RUN_D3_RETEST`);
4. optional and off by default: the generative comparator (switch `RUN_COMPARATOR` in its cell).

Every model is pinned to the revision of its original answers. Answers are written line by line to `My Drive/Jev/paper4/colab/answers_revision/`, so a disconnect loses nothing: run all again and finished requests are skipped. A failing condition or model never stops the others. The last cell lists every missing or incomplete file and writes `answers_revision.zip` next to the folder for hand-back.

In [ ]:
import subprocess as _sp, sys as _sys
_sp.run([_sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, datetime

LOCAL_ROOT = '/content/p4'
SHARED_LOCAL = f'{LOCAL_ROOT}/shared'
SRC_DIR = f'{SHARED_LOCAL}/src'
INPUTS_DIR = f'{SHARED_LOCAL}/colab/inputs'
os.environ['HF_HOME'] = '/content/hf'
os.environ['USE_TF'] = '0'

DRIVE_ROOT = '/content/drive/MyDrive/Jev/paper4/colab'
UPLOAD_ZIP = f'{DRIVE_ROOT}/revision_upload.zip'
ANSWERS_DIR = f'{DRIVE_ROOT}/answers_revision'
os.makedirs(ANSWERS_DIR, exist_ok=True)
RUN_D3_RETEST = True      # rep-2 retest of the open models on the two aligned binary D3 sets
run_log = {'start': datetime.datetime.utcnow().isoformat() + 'Z', 'models': {}}
print('answers on Drive:', ANSWERS_DIR)

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
run_log['gpu'] = gpu_info
print(gpu_info)
if 'L4' not in gpu_info:
    print('WARNING: this is not an L4; latencies will not match the first run.')

## Unpack the upload and verify every file

In [ ]:
import hashlib, zipfile, shutil

EXPECTED = {
 "d1_neutral": {
  "file": "d1_neutral.jsonl",
  "requests": 400,
  "sha256": "013fd3993a5d3e4e77cd267161d6333a202a35ca0daa2342abaf945f8548b591"
 },
 "d2_k150": {
  "file": "d2_k150.jsonl",
  "requests": 800,
  "sha256": "b41706d09f2bd228463d50b5e1cd81d45c7da2fd3abaa5ce7fd869aaabd9a1ec"
 },
 "d2_k150_p2": {
  "file": "d2_k150_p2.jsonl",
  "requests": 600,
  "sha256": "63895898108c914543632b545acbe480ee07018a34c1eb59abc76e97ad161218"
 },
 "d2_k150_p3": {
  "file": "d2_k150_p3.jsonl",
  "requests": 600,
  "sha256": "d7c5116a65aa02b272cc74911555d024062080ffa8ffe5bbefe29b52085a8cc3"
 },
 "d2_k20_p2": {
  "file": "d2_k20_p2.jsonl",
  "requests": 600,
  "sha256": "a9b3641f6901134d5d125b8dbf1ccdb3de8166bd8abe8538c71c0f9eefcfb7a5"
 },
 "d2_k20_p3": {
  "file": "d2_k20_p3.jsonl",
  "requests": 600,
  "sha256": "d6b9d8f31634b9d91fe6bae0c9260ffc28c2c04ecc4ffd2deb75d8befb92b3a3"
 },
 "d2_k5": {
  "file": "d2_k5.jsonl",
  "requests": 600,
  "sha256": "e0dbee0ded6b07ab9fd8b372287561c640f39ac436bb5782bcd24f6bea71d8a2"
 },
 "d2_k50": {
  "file": "d2_k50.jsonl",
  "requests": 600,
  "sha256": "7fd1cc8f33e947e01ca7136df96164ef09b7d5ea0ac65572a948df6ef9dfbb08"
 },
 "d2_k50_p2": {
  "file": "d2_k50_p2.jsonl",
  "requests": 600,
  "sha256": "a69ee019b078160e53954ebfafed27d45c08e702a4d12137b3e04d970b7e56a8"
 },
 "d2_k50_p3": {
  "file": "d2_k50_p3.jsonl",
  "requests": 600,
  "sha256": "065b475d9621fbb18e460f57e1a92ecbbcd68643f432e7ce9429bd0fbe224bc3"
 },
 "d2_k5_p2": {
  "file": "d2_k5_p2.jsonl",
  "requests": 600,
  "sha256": "84ee31be3d6b888201a60429489d81d7a2e5c34ae8353a2f905d097167122e65"
 },
 "d2_k5_p3": {
  "file": "d2_k5_p3.jsonl",
  "requests": 600,
  "sha256": "48228b0bbc8a7b255ab71f7de3598e0b449e7ad98bfa47920894c60a20bdd475"
 },
 "e2_d3_conv_go_awry_kny": {
  "file": "e2_d3_conv_go_awry_kny.jsonl",
  "requests": 500,
  "sha256": "f3615a1592e3aad413ee0fa58d00d63635e9b2439d89aad2ed4755afd66c8c4f"
 },
 "e2_d3_wiki_corpus_kny": {
  "file": "e2_d3_wiki_corpus_kny.jsonl",
  "requests": 500,
  "sha256": "01147e9e891551699ea9bcda03e890758e11aa16e7f90ea6a72b4409ae81eaa6"
 }
}
SRC_SHA256 = {
 "adapters.py": "76aab6f5394b34e392efcc301eca25272ba531e3882902ff11b1358f30246e43",
 "common.py": "7fd1fda9a2914c020818ad036ed088b5c9ffc6eb63346edc5ef084da43d891be",
 "harness.py": "2be4b3216600dc518126c62b1299cb6abe3e6626fb496fc8e4862d1f276af162"
}
ZIP_SHA256 = '15ce4ebda1c73f019586252441e0fd2670b6570a909ad7920687aa3371a39ec9'

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if not os.path.exists(UPLOAD_ZIP):
    raise SystemExit(f'{UPLOAD_ZIP} not found: upload revision_upload.zip to that Drive folder first.')
got = _sha256(UPLOAD_ZIP)
if got != ZIP_SHA256:
    print(f'note: the zip differs from the one this notebook was built with ({got}); '
          'the per-file check below decides')
shutil.rmtree(LOCAL_ROOT, ignore_errors=True)
with zipfile.ZipFile(UPLOAD_ZIP) as z:
    z.extractall(LOCAL_ROOT)
bad = []
for cond, meta in EXPECTED.items():
    p = f'{INPUTS_DIR}/' + meta['file']
    if not os.path.exists(p) or _sha256(p) != meta['sha256']:
        bad.append(cond)
for f, h in SRC_SHA256.items():
    if _sha256(f'{SRC_DIR}/{f}') != h:
        bad.append(f)
if bad:
    raise SystemExit(f'these files do not match the build: {bad} -- stopping.')
print(f'{len(EXPECTED)} input files and {len(SRC_SHA256)} source files verified.')

## Revisions

The commit each model's original answers record. `adapters.py` reads `P4_REVISIONS` and loads exactly these commits instead of the current head.

In [ ]:
REVISIONS = {
 "convaiinnovations/laya": "55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851",
 "jaredpalmer/kev-0.8b": "9a45d25eb2ab761841196625383fa1dff0e56c1e",
 "jaredpalmer/kev-9b": "2629c06a5aeb0feb3b9783bafed17ed8f39ecf5c",
 "Mapika/decider-2b": "d61c1c16089572df5d180329b9fea4997a90090c",
 "flock-io/this-that-model-1.0": "3d927195c4f9845efe66c5715883a7a0f42b1239",
 "bespokelabs/Bespoke-Nimble-9B": "bd792f44ec8e265be861bfcdf4e05967ffe0e858",
 "Qwen/Qwen3-14B-AWQ": "31c69efc29464b6bb0aee1398b5a7b50a99340c3"
}
TAG_REPO = {
 "laya-en": "convaiinnovations/laya",
 "laya-ml": "convaiinnovations/laya",
 "kev-0.8b": "jaredpalmer/kev-0.8b",
 "kev-9b": "jaredpalmer/kev-9b",
 "decider-2b": "Mapika/decider-2b",
 "this-that-1.0": "flock-io/this-that-model-1.0",
 "this-that-1.0-desc": "flock-io/this-that-model-1.0",
 "nimble-9b": "bespokelabs/Bespoke-Nimble-9B",
 "comparator-open": "Qwen/Qwen3-14B-AWQ"
}
print(json.dumps(REVISIONS, indent=1))

## Open decision models

Each cell builds (or reuses) its uv env, smoke-tests 5 requests, prints a time estimate, then runs its conditions one process per condition. Output is streamed and also appended to `answers_revision/<tag>/run_log_<tag>.txt`. Look for `DONE_revision` or `ERROR_revision.txt` in each model folder.

### laya-en

On the K=150 conditions this model refuses every request (its option text exceeds the head budget), exactly as in the first run; those error lines are the expected result, not a failure.

In [ ]:
import subprocess, sys, time, json, os, traceback

TAG = 'laya-en'
ENV_NAME = 'laya'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/laya --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/laya/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 laya==0.3.20 python-dotenv httpx numpy\n'
JOBS = [('laya-en', 'd2_k5_p2', 1), ('laya-en', 'd2_k20_p2', 1), ('laya-en', 'd2_k50_p2', 1), ('laya-en', 'd2_k150_p2', 1), ('laya-en', 'd2_k5_p3', 1), ('laya-en', 'd2_k20_p3', 1), ('laya-en', 'd2_k50_p3', 1), ('laya-en', 'd2_k150_p3', 1)] + ([('laya-en', 'e2_d3_conv_go_awry_kny', 2), ('laya-en', 'e2_d3_wiki_corpus_kny', 2)] if RUN_D3_RETEST else [])            # [(run tag, condition, rep), ...]
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(TAG_DIR, exist_ok=True)
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
SMOKE_DIR = f'/content/smoke/{TAG}'
os.makedirs(SMOKE_DIR, exist_ok=True)

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

failed = []
try:
    print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
    subprocess.run(['bash', '-c', INSTALL], check=True)

    print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
    rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, limit=5, answers_dir=SMOKE_DIR)
    if rc:
        raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
    n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
    print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
          f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
          f'~{dt / 5 * n_req / 3600 * 4.8:.1f} compute units at 4.8/h on an L4 (assumption)', flush=True)
    run_log['models'][TAG] = {'smoke_s_per_request_incl_load': dt / 5, 'requests': n_req}

    for run_tag, cond, rep in JOBS:
        rc, dt = run_harness(run_tag, cond, rep)
        print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min', flush=True)
        if rc:
            failed.append(f'{run_tag}/{cond}__rep{rep}')
    if failed:
        raise RuntimeError(f'{TAG}: conditions failed: {failed}')
    with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print(f'=== {TAG}: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
        fh.write(tb)
    print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision.txt) ===')
    print(tb)
finally:
    import gc
    gc.collect()
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### laya-ml

In [ ]:
import subprocess, sys, time, json, os, traceback

TAG = 'laya-ml'
ENV_NAME = 'laya'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/laya --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/laya/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 laya==0.3.20 python-dotenv httpx numpy\n'
JOBS = [('laya-ml', 'd2_k5_p2', 1), ('laya-ml', 'd2_k20_p2', 1), ('laya-ml', 'd2_k50_p2', 1), ('laya-ml', 'd2_k150_p2', 1), ('laya-ml', 'd2_k5_p3', 1), ('laya-ml', 'd2_k20_p3', 1), ('laya-ml', 'd2_k50_p3', 1), ('laya-ml', 'd2_k150_p3', 1)] + ([('laya-ml', 'e2_d3_conv_go_awry_kny', 2), ('laya-ml', 'e2_d3_wiki_corpus_kny', 2)] if RUN_D3_RETEST else [])            # [(run tag, condition, rep), ...]
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(TAG_DIR, exist_ok=True)
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
SMOKE_DIR = f'/content/smoke/{TAG}'
os.makedirs(SMOKE_DIR, exist_ok=True)

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

failed = []
try:
    print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
    subprocess.run(['bash', '-c', INSTALL], check=True)

    print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
    rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, limit=5, answers_dir=SMOKE_DIR)
    if rc:
        raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
    n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
    print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
          f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
          f'~{dt / 5 * n_req / 3600 * 4.8:.1f} compute units at 4.8/h on an L4 (assumption)', flush=True)
    run_log['models'][TAG] = {'smoke_s_per_request_incl_load': dt / 5, 'requests': n_req}

    for run_tag, cond, rep in JOBS:
        rc, dt = run_harness(run_tag, cond, rep)
        print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min', flush=True)
        if rc:
            failed.append(f'{run_tag}/{cond}__rep{rep}')
    if failed:
        raise RuntimeError(f'{TAG}: conditions failed: {failed}')
    with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print(f'=== {TAG}: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
        fh.write(tb)
    print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision.txt) ===')
    print(tb)
finally:
    import gc
    gc.collect()
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### kev-0.8b

In [ ]:
import subprocess, sys, time, json, os, traceback

TAG = 'kev-0.8b'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('kev-0.8b', 'd2_k5_p2', 1), ('kev-0.8b', 'd2_k20_p2', 1), ('kev-0.8b', 'd2_k50_p2', 1), ('kev-0.8b', 'd2_k150_p2', 1), ('kev-0.8b', 'd2_k5_p3', 1), ('kev-0.8b', 'd2_k20_p3', 1), ('kev-0.8b', 'd2_k50_p3', 1), ('kev-0.8b', 'd2_k150_p3', 1)] + ([('kev-0.8b', 'e2_d3_conv_go_awry_kny', 2), ('kev-0.8b', 'e2_d3_wiki_corpus_kny', 2)] if RUN_D3_RETEST else [])            # [(run tag, condition, rep), ...]
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(TAG_DIR, exist_ok=True)
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
SMOKE_DIR = f'/content/smoke/{TAG}'
os.makedirs(SMOKE_DIR, exist_ok=True)

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

failed = []
try:
    print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
    subprocess.run(['bash', '-c', INSTALL], check=True)

    print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
    rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, limit=5, answers_dir=SMOKE_DIR)
    if rc:
        raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
    n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
    print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
          f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
          f'~{dt / 5 * n_req / 3600 * 4.8:.1f} compute units at 4.8/h on an L4 (assumption)', flush=True)
    run_log['models'][TAG] = {'smoke_s_per_request_incl_load': dt / 5, 'requests': n_req}

    for run_tag, cond, rep in JOBS:
        rc, dt = run_harness(run_tag, cond, rep)
        print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min', flush=True)
        if rc:
            failed.append(f'{run_tag}/{cond}__rep{rep}')
    if failed:
        raise RuntimeError(f'{TAG}: conditions failed: {failed}')
    with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print(f'=== {TAG}: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
        fh.write(tb)
    print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision.txt) ===')
    print(tb)
finally:
    import gc
    gc.collect()
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### decider-2b

In [ ]:
import subprocess, sys, time, json, os, traceback

TAG = 'decider-2b'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('decider-2b', 'd2_k5_p2', 1), ('decider-2b', 'd2_k20_p2', 1), ('decider-2b', 'd2_k50_p2', 1), ('decider-2b', 'd2_k150_p2', 1), ('decider-2b', 'd2_k5_p3', 1), ('decider-2b', 'd2_k20_p3', 1), ('decider-2b', 'd2_k50_p3', 1), ('decider-2b', 'd2_k150_p3', 1)] + ([('decider-2b', 'e2_d3_conv_go_awry_kny', 2), ('decider-2b', 'e2_d3_wiki_corpus_kny', 2)] if RUN_D3_RETEST else [])            # [(run tag, condition, rep), ...]
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(TAG_DIR, exist_ok=True)
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
SMOKE_DIR = f'/content/smoke/{TAG}'
os.makedirs(SMOKE_DIR, exist_ok=True)

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

failed = []
try:
    print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
    subprocess.run(['bash', '-c', INSTALL], check=True)

    print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
    rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, limit=5, answers_dir=SMOKE_DIR)
    if rc:
        raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
    n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
    print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
          f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
          f'~{dt / 5 * n_req / 3600 * 4.8:.1f} compute units at 4.8/h on an L4 (assumption)', flush=True)
    run_log['models'][TAG] = {'smoke_s_per_request_incl_load': dt / 5, 'requests': n_req}

    for run_tag, cond, rep in JOBS:
        rc, dt = run_harness(run_tag, cond, rep)
        print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min', flush=True)
        if rc:
            failed.append(f'{run_tag}/{cond}__rep{rep}')
    if failed:
        raise RuntimeError(f'{TAG}: conditions failed: {failed}')
    with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print(f'=== {TAG}: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
        fh.write(tb)
    print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision.txt) ===')
    print(tb)
finally:
    import gc
    gc.collect()
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### this-that-1.0

Also runs `this-that-1.0-desc`: the same checkpoint with each choice option and score level given as its description alone, on `d1_neutral` and `d2_k150`.

In [ ]:
import subprocess, sys, time, json, os, traceback

TAG = 'this-that-1.0'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f' 'thisthat @ git+https://github.com/FLock-io/this-that-model@542d445efa5f68b14bfbd1f8ed25aacd8379d839'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('this-that-1.0', 'd2_k5_p2', 1), ('this-that-1.0', 'd2_k20_p2', 1), ('this-that-1.0', 'd2_k50_p2', 1), ('this-that-1.0', 'd2_k150_p2', 1), ('this-that-1.0', 'd2_k5_p3', 1), ('this-that-1.0', 'd2_k20_p3', 1), ('this-that-1.0', 'd2_k50_p3', 1), ('this-that-1.0', 'd2_k150_p3', 1), ('this-that-1.0-desc', 'd1_neutral', 1), ('this-that-1.0-desc', 'd2_k150', 1)] + ([('this-that-1.0', 'e2_d3_conv_go_awry_kny', 2), ('this-that-1.0', 'e2_d3_wiki_corpus_kny', 2)] if RUN_D3_RETEST else [])            # [(run tag, condition, rep), ...]
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(TAG_DIR, exist_ok=True)
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
SMOKE_DIR = f'/content/smoke/{TAG}'
os.makedirs(SMOKE_DIR, exist_ok=True)

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

failed = []
try:
    print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
    subprocess.run(['bash', '-c', INSTALL], check=True)

    print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
    rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, limit=5, answers_dir=SMOKE_DIR)
    if rc:
        raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
    n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
    print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
          f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
          f'~{dt / 5 * n_req / 3600 * 4.8:.1f} compute units at 4.8/h on an L4 (assumption)', flush=True)
    run_log['models'][TAG] = {'smoke_s_per_request_incl_load': dt / 5, 'requests': n_req}

    for run_tag, cond, rep in JOBS:
        rc, dt = run_harness(run_tag, cond, rep)
        print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min', flush=True)
        if rc:
            failed.append(f'{run_tag}/{cond}__rep{rep}')
    if failed:
        raise RuntimeError(f'{TAG}: conditions failed: {failed}')
    with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print(f'=== {TAG}: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
        fh.write(tb)
    print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision.txt) ===')
    print(tb)
finally:
    import gc
    gc.collect()
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### kev-9b

In [ ]:
import subprocess, sys, time, json, os, traceback

TAG = 'kev-9b'
ENV_NAME = 'kev_decider_tt'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('kev-9b', 'd2_k5_p2', 1), ('kev-9b', 'd2_k20_p2', 1), ('kev-9b', 'd2_k50_p2', 1), ('kev-9b', 'd2_k150_p2', 1), ('kev-9b', 'd2_k5_p3', 1), ('kev-9b', 'd2_k20_p3', 1), ('kev-9b', 'd2_k50_p3', 1), ('kev-9b', 'd2_k150_p3', 1)] + ([('kev-9b', 'e2_d3_conv_go_awry_kny', 2), ('kev-9b', 'e2_d3_wiki_corpus_kny', 2)] if RUN_D3_RETEST else [])            # [(run tag, condition, rep), ...]
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(TAG_DIR, exist_ok=True)
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
SMOKE_DIR = f'/content/smoke/{TAG}'
os.makedirs(SMOKE_DIR, exist_ok=True)

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

failed = []
try:
    print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
    subprocess.run(['bash', '-c', INSTALL], check=True)

    print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
    rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, limit=5, answers_dir=SMOKE_DIR)
    if rc:
        raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
    n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
    print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
          f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
          f'~{dt / 5 * n_req / 3600 * 4.8:.1f} compute units at 4.8/h on an L4 (assumption)', flush=True)
    run_log['models'][TAG] = {'smoke_s_per_request_incl_load': dt / 5, 'requests': n_req}

    for run_tag, cond, rep in JOBS:
        rc, dt = run_harness(run_tag, cond, rep)
        print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min', flush=True)
        if rc:
            failed.append(f'{run_tag}/{cond}__rep{rep}')
    if failed:
        raise RuntimeError(f'{TAG}: conditions failed: {failed}')
    with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print(f'=== {TAG}: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
        fh.write(tb)
    print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision.txt) ===')
    print(tb)
finally:
    import gc
    gc.collect()
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


### nimble-9b

In [ ]:
import subprocess, sys, time, json, os, traceback

TAG = 'nimble-9b'
ENV_NAME = 'nimble'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/nimble --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/nimble/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 peft==0.21.0 accelerate==1.15.0 sentencepiece==0.2.2 pillow==12.3.0 'huggingface_hub>=0.34' python-dotenv httpx numpy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/nimble/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
JOBS = [('nimble-9b', 'd2_k5_p2', 1), ('nimble-9b', 'd2_k20_p2', 1), ('nimble-9b', 'd2_k50_p2', 1), ('nimble-9b', 'd2_k150_p2', 1), ('nimble-9b', 'd2_k5_p3', 1), ('nimble-9b', 'd2_k20_p3', 1), ('nimble-9b', 'd2_k50_p3', 1), ('nimble-9b', 'd2_k150_p3', 1)] + ([('nimble-9b', 'e2_d3_conv_go_awry_kny', 2), ('nimble-9b', 'e2_d3_wiki_corpus_kny', 2)] if RUN_D3_RETEST else [])            # [(run tag, condition, rep), ...]
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(TAG_DIR, exist_ok=True)
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'
SMOKE_DIR = f'/content/smoke/{TAG}'
os.makedirs(SMOKE_DIR, exist_ok=True)

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

def run_harness(run_tag, cond, rep, limit=None, answers_dir=None):
    """One condition in its own process; every output line goes to the cell and the Drive log."""
    cmd = [PYBIN, f'{SRC_DIR}/harness.py', '--model', run_tag, '--cond', cond, '--rep', str(rep)]
    if limit: cmd += ['--limit', str(limit)]
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=harness_env(answers_dir or ANSWERS_DIR),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='', flush=True)
        rc = p.wait()
    return rc, time.time() - t0

failed = []
try:
    print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
    subprocess.run(['bash', '-c', INSTALL], check=True)

    print(f'=== {TAG}: smoke test (5 requests of {JOBS[0][1]}) ===', flush=True)
    rc, dt = run_harness(JOBS[0][0], JOBS[0][1], 1, limit=5, answers_dir=SMOKE_DIR)
    if rc:
        raise RuntimeError(f'{TAG}: smoke test failed (exit {rc}); see {LOG}')
    n_req = sum(EXPECTED[c]['requests'] for _t, c, _r in JOBS)
    print(f'{TAG}: {dt / 5:.2f} s per request including the model load -> at most '
          f'{dt / 5 * n_req / 60:.0f} min for {n_req} requests, '
          f'~{dt / 5 * n_req / 3600 * 4.8:.1f} compute units at 4.8/h on an L4 (assumption)', flush=True)
    run_log['models'][TAG] = {'smoke_s_per_request_incl_load': dt / 5, 'requests': n_req}

    for run_tag, cond, rep in JOBS:
        rc, dt = run_harness(run_tag, cond, rep)
        print(f'[{run_tag}] {cond} rep{rep}: exit {rc}, {dt / 60:.1f} min', flush=True)
        if rc:
            failed.append(f'{run_tag}/{cond}__rep{rep}')
    if failed:
        raise RuntimeError(f'{TAG}: conditions failed: {failed}')
    with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print(f'=== {TAG}: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
        fh.write(tb)
    print(f'=== {TAG}: ERROR, continuing with the next model (see {TAG_DIR}/ERROR_revision.txt) ===')
    print(tb)
finally:
    import gc
    gc.collect()
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])


## Optional: generative comparator

Off by default. Set `RUN_COMPARATOR = True` in the cell to run Qwen3-14B-AWQ through vLLM on the eight permutation conditions plus `d2_k5` and `d2_k50` (which it did not answer in the first run), so every K has three permutations.

In [ ]:
import subprocess, sys, time, json, os, traceback

RUN_COMPARATOR = False     # set to True to run the comparator (about 1.5 h on an L4)
TAG = 'comparator-open'
ENV_NAME = 'vllm_env'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
CONDS = ['d2_k5_p2', 'd2_k20_p2', 'd2_k50_p2', 'd2_k150_p2', 'd2_k5_p3', 'd2_k20_p3', 'd2_k50_p3', 'd2_k150_p3', 'd2_k5', 'd2_k50']
TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
LOG = f'{TAG_DIR}/run_log_{TAG}.txt'

def stream(cmd, env=None):
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        return p.wait()

def harness_env(answers_dir):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    env['VLLM_LOGGING_LEVEL'] = 'INFO'
    # FlashInfer JIT-compiles kernels with ninja, which lives in the env's bin folder;
    # greedy decoding does not need the FlashInfer sampler, so it is switched off as well
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
    return env

if not RUN_COMPARATOR:
    print('comparator skipped (RUN_COMPARATOR = False)')
else:
    os.makedirs(TAG_DIR, exist_ok=True)
    try:
        print('=== comparator-open: building env ===', flush=True)
        rc = stream(['bash', '-c',
                     'set -e; export UV_CACHE_DIR=/content/uv-cache; '
                     f'uv venv /content/envs/{ENV_NAME} --python 3.12 -q --allow-existing; '
                     f'uv pip install --python {PYBIN} -q '
                     'vllm huggingface_hub python-dotenv httpx numpy ninja; '
                     f'{PYBIN} -c "import vllm, torch; print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'])
        if rc:
            raise RuntimeError(f'env build failed (exit {rc})')
        print('=== comparator-open: smoke test, 5 requests ===', flush=True)
        rc = stream([PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', CONDS[0],
                     '--rep', '1', '--limit', '5'], env=harness_env('/content/smoke/comparator-open'))
        if rc:
            raise RuntimeError(f'smoke test failed (exit {rc}); see {LOG}')
        print('=== comparator-open: full run (one process, model loaded once) ===', flush=True)
        t0 = time.time()
        rc = stream([PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', ','.join(CONDS),
                     '--rep', '1'], env=harness_env(ANSWERS_DIR))
        print(f'full run exit {rc}, {(time.time() - t0) / 60:.1f} min', flush=True)
        if rc:
            raise RuntimeError(f'full run failed (exit {rc}); see {LOG}')
        with open(f'{TAG_DIR}/DONE_revision', 'w') as fh:
            fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
        print('=== comparator-open: DONE ===')
    except Exception:
        tb = traceback.format_exc()
        with open(f'{TAG_DIR}/ERROR_revision.txt', 'w') as fh:
            fh.write(tb)
        print('=== comparator-open: ERROR; the full log is in ' + LOG + ' ===')
        print(tb)


## Status and hand-back

Lists every planned answer file with its count of answered requests and checks the recorded revision against the pin. Then writes `answers_revision.zip` into `My Drive/Jev/paper4/colab/`.

In [ ]:
PLANNED = [('laya-en', 'd2_k5_p2', 1), ('laya-en', 'd2_k20_p2', 1), ('laya-en', 'd2_k50_p2', 1), ('laya-en', 'd2_k150_p2', 1), ('laya-en', 'd2_k5_p3', 1), ('laya-en', 'd2_k20_p3', 1), ('laya-en', 'd2_k50_p3', 1), ('laya-en', 'd2_k150_p3', 1), ('laya-ml', 'd2_k5_p2', 1), ('laya-ml', 'd2_k20_p2', 1), ('laya-ml', 'd2_k50_p2', 1), ('laya-ml', 'd2_k150_p2', 1), ('laya-ml', 'd2_k5_p3', 1), ('laya-ml', 'd2_k20_p3', 1), ('laya-ml', 'd2_k50_p3', 1), ('laya-ml', 'd2_k150_p3', 1), ('kev-0.8b', 'd2_k5_p2', 1), ('kev-0.8b', 'd2_k20_p2', 1), ('kev-0.8b', 'd2_k50_p2', 1), ('kev-0.8b', 'd2_k150_p2', 1), ('kev-0.8b', 'd2_k5_p3', 1), ('kev-0.8b', 'd2_k20_p3', 1), ('kev-0.8b', 'd2_k50_p3', 1), ('kev-0.8b', 'd2_k150_p3', 1), ('decider-2b', 'd2_k5_p2', 1), ('decider-2b', 'd2_k20_p2', 1), ('decider-2b', 'd2_k50_p2', 1), ('decider-2b', 'd2_k150_p2', 1), ('decider-2b', 'd2_k5_p3', 1), ('decider-2b', 'd2_k20_p3', 1), ('decider-2b', 'd2_k50_p3', 1), ('decider-2b', 'd2_k150_p3', 1), ('this-that-1.0', 'd2_k5_p2', 1), ('this-that-1.0', 'd2_k20_p2', 1), ('this-that-1.0', 'd2_k50_p2', 1), ('this-that-1.0', 'd2_k150_p2', 1), ('this-that-1.0', 'd2_k5_p3', 1), ('this-that-1.0', 'd2_k20_p3', 1), ('this-that-1.0', 'd2_k50_p3', 1), ('this-that-1.0', 'd2_k150_p3', 1), ('this-that-1.0-desc', 'd1_neutral', 1), ('this-that-1.0-desc', 'd2_k150', 1), ('kev-9b', 'd2_k5_p2', 1), ('kev-9b', 'd2_k20_p2', 1), ('kev-9b', 'd2_k50_p2', 1), ('kev-9b', 'd2_k150_p2', 1), ('kev-9b', 'd2_k5_p3', 1), ('kev-9b', 'd2_k20_p3', 1), ('kev-9b', 'd2_k50_p3', 1), ('kev-9b', 'd2_k150_p3', 1), ('nimble-9b', 'd2_k5_p2', 1), ('nimble-9b', 'd2_k20_p2', 1), ('nimble-9b', 'd2_k50_p2', 1), ('nimble-9b', 'd2_k150_p2', 1), ('nimble-9b', 'd2_k5_p3', 1), ('nimble-9b', 'd2_k20_p3', 1), ('nimble-9b', 'd2_k50_p3', 1), ('nimble-9b', 'd2_k150_p3', 1)]
PLANNED_D3 = [('laya-en', 'e2_d3_conv_go_awry_kny', 2), ('laya-en', 'e2_d3_wiki_corpus_kny', 2), ('laya-ml', 'e2_d3_conv_go_awry_kny', 2), ('laya-ml', 'e2_d3_wiki_corpus_kny', 2), ('kev-0.8b', 'e2_d3_conv_go_awry_kny', 2), ('kev-0.8b', 'e2_d3_wiki_corpus_kny', 2), ('decider-2b', 'e2_d3_conv_go_awry_kny', 2), ('decider-2b', 'e2_d3_wiki_corpus_kny', 2), ('this-that-1.0', 'e2_d3_conv_go_awry_kny', 2), ('this-that-1.0', 'e2_d3_wiki_corpus_kny', 2), ('kev-9b', 'e2_d3_conv_go_awry_kny', 2), ('kev-9b', 'e2_d3_wiki_corpus_kny', 2), ('nimble-9b', 'e2_d3_conv_go_awry_kny', 2), ('nimble-9b', 'e2_d3_wiki_corpus_kny', 2)]
PLANNED_COMPARATOR = [('comparator-open', 'd2_k5_p2', 1), ('comparator-open', 'd2_k20_p2', 1), ('comparator-open', 'd2_k50_p2', 1), ('comparator-open', 'd2_k150_p2', 1), ('comparator-open', 'd2_k5_p3', 1), ('comparator-open', 'd2_k20_p3', 1), ('comparator-open', 'd2_k50_p3', 1), ('comparator-open', 'd2_k150_p3', 1), ('comparator-open', 'd2_k5', 1), ('comparator-open', 'd2_k50', 1)]
import json, os, datetime

def n_ok_lines(path):
    ok = err = 0
    revs, models = set(), set()
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            try:
                j = json.loads(line)
            except json.JSONDecodeError:
                continue
            if j.get('error'):
                err += 1
            else:
                ok += 1
                revs.add(j.get('revision')); models.add(j.get('model'))
    return ok, err, revs, models

expected = list(PLANNED)
if RUN_D3_RETEST:
    expected += PLANNED_D3
if globals().get('RUN_COMPARATOR'):
    expected += PLANNED_COMPARATOR
status, problems = {}, []
for tag, cond, rep in expected:
    path = f'{ANSWERS_DIR}/{tag}/{cond}__rep{rep}.jsonl'
    want = EXPECTED[cond]['requests']
    if not os.path.exists(path):
        status[f'{tag}/{cond}__rep{rep}'] = 'MISSING'
        problems.append(path)
        continue
    ok, err, revs, models = n_ok_lines(path)
    pin = REVISIONS[TAG_REPO[tag]]
    rev_ok = revs == {pin}
    note = f'{ok}/{want} answered, {err} error lines, revision {"ok" if rev_ok else revs}'
    # Laya's English head refuses K=150 by design (its option text exceeds the head budget):
    # error lines there are the expected outcome, recorded exactly as in the first run
    expected_refusal = tag == 'laya-en' and cond.startswith('d2_k150')
    complete = (ok + err >= want) if expected_refusal else (ok == want)
    status[f'{tag}/{cond}__rep{rep}'] = ('OK ' if complete and (rev_ok or expected_refusal and ok == 0)
                                         else 'INCOMPLETE ') + note
    if not (complete and (rev_ok or expected_refusal and ok == 0)):
        problems.append(path)

for k, v in status.items():
    print(f'{k:55s} {v}')
run_log['end'] = datetime.datetime.utcnow().isoformat() + 'Z'
run_log['status'] = status
run_log['revisions_pinned'] = REVISIONS
with open(f'{ANSWERS_DIR}/run_log_revision.json', 'w') as fh:
    json.dump(run_log, fh, indent=2)

if problems:
    print(f'\n{len(problems)} file(s) missing or incomplete. Re-run the cell of that model: '
          'finished requests are skipped, only the rest is computed.')
else:
    with open(f'{ANSWERS_DIR}/ALL_DONE_revision', 'w') as fh:
        fh.write(run_log['end'] + '\n')
    print('\nEvery planned answer file is complete. ALL_DONE_revision written.')

# the hand-back archive, next to the folder on Drive: the answer files, the DONE markers and
# run_log_revision.json; the per-model console logs and tracebacks stay on Drive only
import zipfile
archive = f'{DRIVE_ROOT}/answers_revision.zip'
n_files = 0
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _dirs, files in os.walk(ANSWERS_DIR):
        for fn in sorted(files):
            if fn.endswith('.jsonl') or fn in ('DONE_revision', 'run_log_revision.json',
                                               'ALL_DONE_revision'):
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, ANSWERS_DIR))
                n_files += 1
print(f'hand-back archive: {archive} ({n_files} files)')
